### Realtime Conversation with Voice Translation: Part 1 Eng to Spa
#### ` Description: Please create a machine learning model that facilitates real-time conversation between an English-speaking person and a Spanish-speaking person. The model should: Extract Spanish words from voice input and translate them into English, then read the translated word aloud. Similarly, take English voice input from the other user, translate it into Spanish, and read the translated word aloud. Guidelines: Make your own machine learning model. GUI is not mandatory for this. This task is a tough one. Don’t worry, accuracy doesn’t matter. The only thing matter is the amount of effort you have put. The evaluation will be conducted on models’ overall performance.`

## Extending the GPU memory

In [1]:
import tensorflow as tf
# GPU memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)


## Load all the Libraries

In [1]:
import numpy as np
import pandas as pd
import pickle

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Concatenate
from tensorflow.keras.models import Model
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction



## Load the dataset

In [2]:
df = pd.read_csv("eng_spa_small.csv")   
# lowercase + strip
df["en"] = df["en"].str.lower().str.strip()
df["es"] = df["es"].str.lower().str.strip()

# add start & end tokens to target
df["es"] = "<start> " + df["es"] + " <end>"
df

,en,es
0,it's only three days till christmas.,<start> solo faltan tres días para navidad. <end>
1,harvard was founded in 1636.,<start> harvard se fundó en 1636. <end>
2,why do you waste most of your time on tatoeba?,<start> ¿por qué desperdicias la mayoría del t...
3,it is magnificent indeed.,<start> eso sí que es grandioso. <end>
4,she traveled all over the world.,<start> ella viajó por todo el mundo. <end>
...,...,...
14995,do you like chocolate milk?,<start> ¿te gusta la leche con chocolate? <end>
14996,tom set a trap.,<start> tom me puso una trampa. <end>
14997,i'll never eat it again.,<start> no lo volveré a comer. <end>
14998,i sweat every day.,<start> sudo todos los días. <end>


## Preprocess the dataset

In [24]:
eng_tokenizer = Tokenizer(filters="")
spa_tokenizer = Tokenizer(filters="")

eng_tokenizer.fit_on_texts(df["en"])
spa_tokenizer.fit_on_texts(df["es"])

eng_seq = eng_tokenizer.texts_to_sequences(df["en"])
spa_seq = spa_tokenizer.texts_to_sequences(df["es"])


In [25]:
MAX_ENG = max(len(seq) for seq in eng_seq)
MAX_SPA = max(len(seq) for seq in spa_seq)

eng_seq = pad_sequences(eng_seq, maxlen=MAX_ENG, padding="post")
spa_seq = pad_sequences(spa_seq, maxlen=MAX_SPA, padding="post")

ENG_VOCAB = len(eng_tokenizer.word_index) + 1
SPA_VOCAB = len(spa_tokenizer.word_index) + 1


In [26]:
decoder_input = spa_seq[:, :-1]
decoder_target = spa_seq[:, 1:]
decoder_target = np.expand_dims(decoder_target, -1)


## Build the model
1. Build attention layer
2. Build the encoder decoder layer

In [5]:
import tensorflow as tf
from tensorflow.keras.layers import Dense

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, enc_out, dec_out):
        score = self.V(tf.nn.tanh(
            tf.expand_dims(self.W1(enc_out), 1) +
            tf.expand_dims(self.W2(dec_out), 2)
        ))
        attention_weights = tf.nn.softmax(score, axis=2)
        context = attention_weights * tf.expand_dims(enc_out, 1)
        context = tf.reduce_sum(context, axis=2)
        return context

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config


In [9]:
EMBED_DIM = 256
LATENT_DIM = 256


In [10]:
encoder_inputs = Input(shape=(MAX_ENG,))
enc_emb = Embedding(ENG_VOCAB, EMBED_DIM, mask_zero=True)(encoder_inputs)
enc_out, enc_h, enc_c = LSTM(
    LATENT_DIM, return_sequences=True, return_state=True
)(enc_emb)


In [11]:
decoder_inputs = Input(shape=(MAX_SPA - 1,))
dec_emb = Embedding(SPA_VOCAB, EMBED_DIM, mask_zero=True)(decoder_inputs)

dec_out, _, _ = LSTM(
    LATENT_DIM, return_sequences=True, return_state=True
)(dec_emb, initial_state=[enc_h, enc_c])


In [12]:
attention = BahdanauAttention(LATENT_DIM)
context = attention(enc_out, dec_out)

concat = Concatenate(axis=-1)([dec_out, context])
outputs = Dense(SPA_VOCAB, activation="softmax")(concat)


## Compile and Train the model

In [13]:
model = Model([encoder_inputs, decoder_inputs], outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 19)]         0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 20)]         0           []                               
                                                                                                  
 embedding (Embedding)          (None, 19, 256)      3064320     ['input_1[0][0]']                
                                                                                                  
 embedding_1 (Embedding)        (None, 20, 256)      4219392     ['input_2[0][0]']                
                                                                                              

In [16]:
model.fit(
    [eng_seq, decoder_input],
    decoder_target,
    batch_size=64,
    epochs=20,
    validation_split=0.1
)


Epoch 1/20
211/211 [==============================] - 60s 283ms/step - loss: 2.5514 - accuracy: 0.2295 - val_loss: 2.3171 - val_accuracy: 0.2680
Epoch 2/20
211/211 [==============================] - 52s 245ms/step - loss: 2.1316 - accuracy: 0.2813 - val_loss: 2.2236 - val_accuracy: 0.2893
Epoch 3/20
211/211 [==============================] - 52s 245ms/step - loss: 1.9533 - accuracy: 0.3169 - val_loss: 2.1115 - val_accuracy: 0.3329
Epoch 4/20
211/211 [==============================] - 92s 437ms/step - loss: 1.7557 - accuracy: 0.3650 - val_loss: 2.0184 - val_accuracy: 0.3641
Epoch 5/20
211/211 [==============================] - 61s 287ms/step - loss: 1.5629 - accuracy: 0.4074 - val_loss: 1.9460 - val_accuracy: 0.3925
Epoch 6/20
211/211 [==============================] - 49s 231ms/step - loss: 1.3713 - accuracy: 0.4469 - val_loss: 1.9038 - val_accuracy: 0.4112
Epoch 7/20
211/211 [==============================] - 43s 206ms/step - loss: 1.1796 - accuracy: 0.4850 - val_loss: 1.8924 - val_ac

## Save the model and tokenizers

In [27]:
model.save("eng_to_spa_translation.h5")

with open("eng_tokenizer_2.pkl", "wb") as f:
    pickle.dump(eng_tokenizer, f)

with open("spa_tokenizer_2.pkl", "wb") as f:
    pickle.dump(spa_tokenizer, f)

with open("config_eng_spa.pkl", "wb") as f:
    pickle.dump({
        "MAX_ENG": MAX_ENG,
        "MAX_SPA": MAX_SPA,
        "ENG_VOCAB": ENG_VOCAB,
        "SPA_VOCAB": SPA_VOCAB
    }, f)


## Load the model and tokenizers

In [6]:
model = tf.keras.models.load_model(
    "eng_to_spa_translation.h5",
    custom_objects={"BahdanauAttention": BahdanauAttention},
    compile=False
)

with open("eng_tokenizer.pkl", "rb") as f:
    eng_tokenizer = pickle.load(f)

with open("spa_tokenizer.pkl", "rb") as f:
    spa_tokenizer = pickle.load(f)

with open("config_eng_spa.pkl", "rb") as f:
    config = pickle.load(f)

MAX_ENG = config["MAX_ENG"]
MAX_SPA = config["MAX_SPA"]


## Build a translate function

In [8]:
def translate_eng_to_spa(sentence):
    # Encode input
    seq = eng_tokenizer.texts_to_sequences([sentence.lower()])
    seq = tf.keras.preprocessing.sequence.pad_sequences(
        seq, maxlen=MAX_ENG, padding="post"
    )

    # Start token
    start_id = spa_tokenizer.word_index["<start>"]
    end_id   = spa_tokenizer.word_index["<end>"]

    decoder_input = np.zeros((1, MAX_SPA - 1))
    decoder_input[0, 0] = start_id

    result = []

    for t in range(1, MAX_SPA - 1):
        pred = model.predict([seq, decoder_input], verbose=0)
        pred_id = np.argmax(pred[0, t-1])

        if pred_id == end_id:
            break

        result.append(pred_id)
        decoder_input[0, t] = pred_id

    # Convert ids to words
    rev_spa = {v: k for k, v in spa_tokenizer.word_index.items()}
    words = [rev_spa.get(i, "") for i in result]

    return " ".join(words)


## Make some predictions

In [12]:
 test_sentences = df.sample(5)
for src, act  in zip (test_sentences['en'], test_sentences['es']):
    pred = translate_eng_to_spa(src)
   

    print(f"\nSRC: {src}")
    print(f"PRED: {pred}")
    print(f"ACTUAL : {act}")




SRC: Tom put his phone in his pocket.
PRED: tom puso su teléfono en su bolsillo.
ACTUAL : Tom puso su teléfono en su bolsillo.

SRC: He is slowly catching up.
PRED: el está lentamente alcanzándole.
ACTUAL : El está lentamente alcanzándole.

SRC: Don't you get bored when you're alone?
PRED: ¿no te aburres cuando estás sola?
ACTUAL : ¿No te aburres cuando estás sola?

SRC: He always does the opposite of what I tell him to do.
PRED: él siempre hace lo opuesto de lo que le digo.
ACTUAL : Él siempre hace lo opuesto de lo que le digo.

SRC: You're cool.
PRED: sos un langa.
ACTUAL : Sos un langa.
